# 00 · Preparación de datos (Lending Club)

**Autores:** Santiago Hurtado, Juan Marín, Andrés Parejo

Ejecutado **localmente** (PC propio, no Google Colab): el CSV crudo ya
estaba descargado en disco de una corrida anterior de Kaggle, así que este
notebook parte directo del escaneo completo de las 151 columnas originales
y produce el Parquet limpio de ~24 columnas justificadas que reutilizan
**todos** los notebooks siguientes (EDA, preprocesamiento sklearn, LIME).
El preprocesamiento de **PySpark** vuelve a leer el CSV crudo de forma
independiente (notebook 03), tal como exige el enunciado.

**Condición obligatoria del proyecto:** no se muestrea ni se reduce el
número de filas en ningún momento. Las 2,260,701 filas originales se
mantienen íntegras; lo único que se reduce aquí es el número de *columnas*
leídas (de 151 a ~24), lo cual el propio enunciado permite explícitamente
en la sección de preprocesamiento ("Seleccionar variables relevantes").

## 1. Cargar `utils.py` local

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
import utils
import importlib
importlib.reload(utils)
print("utils.py listo en:", utils.LOCAL_ROOT)
print(utils.get_system_resources())

utils.py listo en: C:\Users\boolean\Downloads\requirements fin\Copy of requirements
{'cpu_count': 12, 'total_ram_gb': 63.3, 'free_ram_gb': 47.12}


## 2. Verificar el CSV crudo (ya cacheado en disco local)

In [2]:
assert os.path.exists(utils.RAW_CSV_LOCAL), f"No se encontró el CSV en {utils.RAW_CSV_LOCAL}"
size_gb = os.path.getsize(utils.RAW_CSV_LOCAL) / 1e9
print(f"CSV local listo: {utils.RAW_CSV_LOCAL} ({size_gb:.2f} GB)")

CSV local listo: C:\Users\boolean\Downloads\archive\accepted_2007_to_2018q4.csv\accepted_2007_to_2018Q4.csv (1.68 GB)


## 3. Escaneo completo de las 151 columnas originales

Satisface literalmente el requisito de "cargar el dataset completo... `info()`,
`describe()`, dimensión" **antes** de reducir a las columnas de trabajo. Se
hace en chunks de 200,000 filas para no materializar 151 columnas × 2.26M
filas en memoria a la vez.

In [3]:
import time
import pandas as pd

t0 = time.time()
scan = utils.full_column_scan(utils.RAW_CSV_LOCAL)
t_scan = time.time() - t0

utils.save_json(scan, utils.FULL_SCAN_JSON_DRIVE)

print(f"Filas: {scan['n_rows']:,} | Columnas: {scan['n_cols']} | tiempo de escaneo: {t_scan:.1f}s")

Filas: 2,260,701 | Columnas: 151 | tiempo de escaneo: 40.8s


In [4]:
null_pct_full = pd.Series(scan["null_pct"]).sort_values(ascending=False)
null_pct_full.to_frame("% nulos").head(25)

,% nulos
member_id,100.000
orig_projected_additional_accrued_interest,99.617
hardship_reason,99.517
hardship_payoff_balance_amount,99.517
hardship_last_payment_amount,99.517
payment_plan_start_date,99.517
hardship_type,99.517
hardship_status,99.517
hardship_start_date,99.517
deferral_term,99.517


De las 151 columnas originales, el bloque de arriba muestra las 25 con más
valores faltantes. Muchas superan 70–90% de nulos: son campos de
*hardship*/reestructuración (`hardship_*`, `settlement_*`), variables de
co-solicitante (`sec_app_*`, `annual_inc_joint`) que solo aplican a
préstamos conjuntos, o campos legados que Lending Club dejó de reportar en
años tempranos. Ninguna de estas entra al subconjunto de 24 columnas de
trabajo (ver celda siguiente) — por eso no se necesita una estrategia de
imputación agresiva para ellas.

## 4. Selección de columnas de trabajo (justificación)

De las 151 columnas se seleccionan **24 + el target**, bajo dos criterios:

1. **Disponibilidad al momento de originar el préstamo.** Se excluyen ~127
   columnas post-originación (`total_pymnt`, `recoveries`, `out_prncp`,
   `last_pymnt_d`, `hardship_*`, `settlement_*`, `last_fico_range_*`, etc.)
   porque solo se conocen **después** de que el préstamo empieza a pagarse
   o a incumplir — usarlas sería fuga de información (*data leakage*): el
   modelo "adivinaría" el resultado en vez de predecirlo.
2. **Relevancia explícita del enunciado** (`loan_amnt`, `int_rate`,
   `annual_inc`, `dti`, `fico_range_high`, `emp_length`, `purpose`,
   `home_ownership`, `addr_state`, `verification_status`) más predictores
   estándar de riesgo crediticio disponibles en el momento de la solicitud
   (`installment`, `revol_util`, `revol_bal`, `open_acc`, `total_acc`,
   `mort_acc`, `pub_rec`, `delinq_2yrs`, `term`, `grade`,
   `initial_list_status`, `application_type`).

Esto **no** es muestreo de filas — las 2,260,701 filas permanecen intactas;
solo se reduce cuántas *columnas* de cada fila se cargan en memoria, algo
que el propio enunciado permite ("Seleccionar variables relevantes").

In [5]:
df, meta = utils.build_clean_dataframe(utils.RAW_CSV_LOCAL)
print(meta)
df.info()

{'n_before': 2260701, 'n_after': 2260668, 'n_dropped_blank_rows': 33}
<class 'pandas.core.frame.DataFrame'>
Index: 2260668 entries, 0 to 2260698
Data columns (total 30 columns):
 #   Column               Dtype         
---  ------               -----         
 0   loan_amnt            float32       
 1   term                 category      
 2   int_rate             float32       
 3   installment          float32       
 4   grade                category      
 5   sub_grade            category      
 6   emp_length           category      
 7   home_ownership       category      
 8   annual_inc           float32       
 9   verification_status  category      
 10  issue_d              object        
 11  loan_status          category      
 12  purpose              category      
 13  addr_state           category      
 14  dti                  float32       
 15  delinq_2yrs          float32       
 16  fico_range_low       float32       
 17  fico_range_high      float32       
 1

`n_dropped_blank_rows` corresponde a un puñado de filas (~33 de 2,260,701 =
0.0015%) completamente vacías al final del CSV — sin `loan_status` ni
ningún otro campo. Se excluyen porque no tienen target utilizable (no se
puede entrenar ni evaluar con una fila sin etiqueta); no es una forma de
muestreo, es descarte de filas corruptas/sin información.

In [6]:
df.head()

,loan_amnt,term,int_rate,installment,grade,sub_grade,emp_length,home_ownership,annual_inc,verification_status,...,revol_bal,revol_util,total_acc,initial_list_status,application_type,mort_acc,emp_length_num,issue_d_parsed,issue_year,default
0,3600.0,36 months,13.990000,123.029999,C,C4,10+ years,MORTGAGE,55000.0,Not Verified,...,2765.0,29.700001,13.0,w,Individual,1.0,10.0,2015-12-01,2015,0
1,24700.0,36 months,11.990000,820.280029,C,C1,10+ years,MORTGAGE,65000.0,Not Verified,...,21470.0,19.200001,38.0,w,Individual,4.0,10.0,2015-12-01,2015,0
2,20000.0,60 months,10.780000,432.660004,B,B4,10+ years,MORTGAGE,63000.0,Not Verified,...,7869.0,56.200001,18.0,w,Joint App,5.0,10.0,2015-12-01,2015,0
3,35000.0,60 months,14.850000,829.900024,C,C5,10+ years,MORTGAGE,110000.0,Source Verified,...,7802.0,11.600000,17.0,w,Individual,1.0,10.0,2015-12-01,2015,0
4,10400.0,60 months,22.450001,289.910004,F,F1,3 years,MORTGAGE,104433.0,Source Verified,...,21929.0,64.500000,35.0,w,Individual,6.0,3.0,2015-12-01,2015,0


In [7]:
df.tail()

,loan_amnt,term,int_rate,installment,grade,sub_grade,emp_length,home_ownership,annual_inc,verification_status,...,revol_bal,revol_util,total_acc,initial_list_status,application_type,mort_acc,emp_length_num,issue_d_parsed,issue_year,default
2260694,24000.0,60 months,12.79,543.500000,C,C1,7 years,MORTGAGE,95000.0,Source Verified,...,49431.0,84.400002,54.0,f,Individual,0.0,7.0,2016-10-01,2016,0
2260695,24000.0,60 months,10.49,515.739990,B,B3,10+ years,MORTGAGE,108000.0,Not Verified,...,21665.0,39.000000,58.0,f,Individual,4.0,10.0,2016-10-01,2016,0
2260696,40000.0,60 months,10.49,859.559998,B,B3,9 years,MORTGAGE,227000.0,Verified,...,8633.0,64.900002,37.0,f,Individual,3.0,9.0,2016-10-01,2016,0
2260697,24000.0,60 months,14.49,564.559998,C,C4,6 years,RENT,110000.0,Not Verified,...,17641.0,68.099998,31.0,f,Individual,2.0,6.0,2016-10-01,2016,1
2260698,14000.0,60 months,14.49,329.329987,C,C4,10+ years,MORTGAGE,95000.0,Verified,...,7662.0,54.000000,22.0,w,Individual,1.0,10.0,2016-10-01,2016,0


In [8]:
print("Distribución de loan_status (previo a colapsar en `default`):")
df["loan_status"].value_counts()

Distribución de loan_status (previo a colapsar en `default`):


loan_status
Fully Paid                                             1076751
Current                                                 878317
Charged Off                                             268559
Late (31-120 days)                                       21467
In Grace Period                                           8436
Late (16-30 days)                                         4349
Does not meet the credit policy. Status:Fully Paid        1988
Does not meet the credit policy. Status:Charged Off        761
Default                                                     40
Name: count, dtype: int64

In [9]:
print("Distribución de la variable objetivo `default`:")
print(df["default"].value_counts())
print((df["default"].value_counts(normalize=True) * 100).round(2))

Distribución de la variable objetivo `default`:
default
0    1992109
1     268559
Name: count, dtype: int64
default
0    88.12
1    11.88
Name: proportion, dtype: float64


## 5. Guardar el dataset limpio localmente

Parquet conserva los dtypes (`category`, `float32`) y es mucho más rápido
de releer que reparsear el CSV de 1.67GB en cada notebook siguiente.

In [10]:
t0 = time.time()
df.to_parquet(utils.CLEAN_PARQUET_DRIVE, index=False)
t_save = time.time() - t0

utils.save_json(
    {**meta, "save_parquet_seconds": round(t_save, 2), "n_rows_final": len(df), "n_cols_final": df.shape[1]},
    f"{utils.RESULTS_DIR}/00_clean_meta.json",
)
print(f"Parquet guardado en {t_save:.1f}s ->", utils.CLEAN_PARQUET_DRIVE)

Parquet guardado en 1.8s -> C:\Users\boolean\Downloads\requirements fin\Copy of requirements/data/clean_full.parquet


## 6. Verificación final

In [11]:
df_check = pd.read_parquet(utils.CLEAN_PARQUET_DRIVE)
assert df_check.shape[0] == len(df), "Filas no coinciden tras guardar/releer"
assert df_check.shape[0] > 1_300_000, "Menos de 1.3M filas -- viola la condicion obligatoria del dataset completo"
print("OK ->", df_check.shape, "· filas >= 1.3M:", df_check.shape[0] > 1_300_000)
df_check.dtypes

OK -> (2260668, 30) · filas >= 1.3M: True


loan_amnt                     float32
term                         category
int_rate                      float32
installment                   float32
grade                        category
sub_grade                    category
emp_length                   category
home_ownership               category
annual_inc                    float32
verification_status          category
issue_d                        object
loan_status                  category
purpose                      category
addr_state                   category
dti                           float32
delinq_2yrs                   float32
fico_range_low                float32
fico_range_high               float32
open_acc                      float32
pub_rec                       float32
revol_bal                     float32
revol_util                    float32
total_acc                     float32
initial_list_status          category
application_type             category
mort_acc                      float32
emp_length_n